In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("Tasks").master("local[*]") \
    .config("spark.python.worker.reuse","true").getOrCreate()

data = [("ESP32 WROOM","Microcontroller",350,500),
        ("STM32 Blue Pill","Microcontroller",280,300),
        ("DHT22 Sensor","Sensor",120,800),
        ("PMS5003","Sensor",950,150),
        ("LoRa Ra-02","Communication",320,250),
        ("OLED Display","Display",95,700),
        ("SIM7600 4G","Communication",1200,80),
        ("18650 Battery","Power",220,1000)]
columns = ["product_name","category","price","stock"]
df = spark.createDataFrame(data, columns)
print("✅ Ready!")

✅ Ready!


In [2]:
df.show()

+---------------+---------------+-----+-----+
|   product_name|       category|price|stock|
+---------------+---------------+-----+-----+
|    ESP32 WROOM|Microcontroller|  350|  500|
|STM32 Blue Pill|Microcontroller|  280|  300|
|   DHT22 Sensor|         Sensor|  120|  800|
|        PMS5003|         Sensor|  950|  150|
|     LoRa Ra-02|  Communication|  320|  250|
|   OLED Display|        Display|   95|  700|
|     SIM7600 4G|  Communication| 1200|   80|
|  18650 Battery|          Power|  220| 1000|
+---------------+---------------+-----+-----+



In [4]:
df.select("product_name","price").show()

##df.select("product_name", "price").show()

+---------------+-----+
|   product_name|price|
+---------------+-----+
|    ESP32 WROOM|  350|
|STM32 Blue Pill|  280|
|   DHT22 Sensor|  120|
|        PMS5003|  950|
|     LoRa Ra-02|  320|
|   OLED Display|   95|
|     SIM7600 4G| 1200|
|  18650 Battery|  220|
+---------------+-----+



In [5]:
df.filter(df["price"] > 300).show()

+------------+---------------+-----+-----+
|product_name|       category|price|stock|
+------------+---------------+-----+-----+
| ESP32 WROOM|Microcontroller|  350|  500|
|     PMS5003|         Sensor|  950|  150|
|  LoRa Ra-02|  Communication|  320|  250|
|  SIM7600 4G|  Communication| 1200|   80|
+------------+---------------+-----+-----+



In [6]:
from pyspark.sql.functions import col
df.filter(col("price") > 300).show()

+------------+---------------+-----+-----+
|product_name|       category|price|stock|
+------------+---------------+-----+-----+
| ESP32 WROOM|Microcontroller|  350|  500|
|     PMS5003|         Sensor|  950|  150|
|  LoRa Ra-02|  Communication|  320|  250|
|  SIM7600 4G|  Communication| 1200|   80|
+------------+---------------+-----+-----+



In [7]:
df.orderBy("price", ascending=False).show()

df.orderBy("price", ascending=False).show()

+---------------+---------------+-----+-----+
|   product_name|       category|price|stock|
+---------------+---------------+-----+-----+
|     SIM7600 4G|  Communication| 1200|   80|
|        PMS5003|         Sensor|  950|  150|
|    ESP32 WROOM|Microcontroller|  350|  500|
|     LoRa Ra-02|  Communication|  320|  250|
|STM32 Blue Pill|Microcontroller|  280|  300|
|  18650 Battery|          Power|  220| 1000|
|   DHT22 Sensor|         Sensor|  120|  800|
|   OLED Display|        Display|   95|  700|
+---------------+---------------+-----+-----+

+---------------+---------------+-----+-----+
|   product_name|       category|price|stock|
+---------------+---------------+-----+-----+
|     SIM7600 4G|  Communication| 1200|   80|
|        PMS5003|         Sensor|  950|  150|
|    ESP32 WROOM|Microcontroller|  350|  500|
|     LoRa Ra-02|  Communication|  320|  250|
|STM32 Blue Pill|Microcontroller|  280|  300|
|  18650 Battery|          Power|  220| 1000|
|   DHT22 Sensor|         Sensor|

In [8]:
df.count()

8

In [9]:
print("Total products:", df.count())

Total products: 8


In [11]:
df.withColumn("stock_value", df["price"] * df["stock"]).show()

+---------------+---------------+-----+-----+-----------+
|   product_name|       category|price|stock|stock_value|
+---------------+---------------+-----+-----+-----------+
|    ESP32 WROOM|Microcontroller|  350|  500|     175000|
|STM32 Blue Pill|Microcontroller|  280|  300|      84000|
|   DHT22 Sensor|         Sensor|  120|  800|      96000|
|        PMS5003|         Sensor|  950|  150|     142500|
|     LoRa Ra-02|  Communication|  320|  250|      80000|
|   OLED Display|        Display|   95|  700|      66500|
|     SIM7600 4G|  Communication| 1200|   80|      96000|
|  18650 Battery|          Power|  220| 1000|     220000|
+---------------+---------------+-----+-----+-----------+



In [12]:
df.filter(col("category") == "Sensor").show()

+------------+--------+-----+-----+
|product_name|category|price|stock|
+------------+--------+-----+-----+
|DHT22 Sensor|  Sensor|  120|  800|
|     PMS5003|  Sensor|  950|  150|
+------------+--------+-----+-----+



In [13]:
df.groupBy("category").count().orderBy("count", ascending=False).show()

df.groupby("category").count().orderBy("count", ascending=False).show()

+---------------+-----+
|       category|count|
+---------------+-----+
|Microcontroller|    2|
|         Sensor|    2|
|  Communication|    2|
|        Display|    1|
|          Power|    1|
+---------------+-----+

+---------------+-----+
|       category|count|
+---------------+-----+
|Microcontroller|    2|
|         Sensor|    2|
|  Communication|    2|
|        Display|    1|
|          Power|    1|
+---------------+-----+



In [14]:
spark

In [15]:
df.groupBy("category") \
  .agg(F.avg("price").alias("avg_price"),
       F.sum("stock").alias("total_stock")) \
  .orderBy("avg_price", ascending=False) \
  .show()

+---------------+---------+-----------+
|       category|avg_price|total_stock|
+---------------+---------+-----------+
|  Communication|    760.0|        330|
|         Sensor|    535.0|        950|
|Microcontroller|    315.0|        800|
|          Power|    220.0|       1000|
|        Display|     95.0|        700|
+---------------+---------+-----------+

